In [1]:
import streamlit as st
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi import HTTPException
from fastapi.responses import JSONResponse
from contextlib import asynccontextmanager
import nest_asyncio
import uvicorn
import threading
import mlflow
from openai import OpenAI, OpenAIError
from loguru import logger

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

/opt/homebrew/Caskroom/miniconda/base/envs/agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the MLflow experiment parameters
PORT = "5001"
EXPERIMENT_NAME = "FastAPI-Ollama"
OLLAMA_API_URL = "http://localhost:11434/v1"
# Define LLM model and parameters
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [3]:
# MFlow autologging setup
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774188582234, experiment_id='1', last_update_time=1774188582234, lifecycle_stage='active', name='FastAPI-Ollama', tags={}, workspace='default'>

In [4]:
def _build_client(base_url: str) -> OpenAI:
    """
    Set up the OpenAI client to connect to the Ollama API.
    """
    if not base_url:
        raise ValueError("OLLAMA_API_URL is not set")
    logger.info(f"Initializing OpenAI client with base_url={base_url!r}")
    return OpenAI(
        base_url=base_url,
        api_key="dummy",  # Ollama doesn't require a real key
    )

client = _build_client(OLLAMA_API_URL)

2026-03-24 09:53:45.741 | INFO     | __main__:_build_client:7 - Initializing OpenAI client with base_url='http://localhost:11434/v1'


In [5]:
def llm_generate(prompt: str):
    try:
        logger.info(f"Generating response for prompt: {prompt}")
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        content_response = response.choices[0].message.content
        logger.info(f"Generated response: {content_response}")
        return content_response
    except OpenAIError as e:
        logger.error(f"Error generating response: {e}")
        raise HTTPException(status_code=500, detail="Error generating response from LLM")

In [6]:
class PromptRequest(BaseModel):
    prompt: str

In [7]:
app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "World"}

@app.post("/generate")
def generate_response(request: PromptRequest):
    try:
        response = llm_generate(request.prompt)
        return {"response": response}
    except Exception as e:
        logger.error(f"Error in /generate endpoint: {e}")
        raise HTTPException(status_code=500, detail="Internal Server Error")
        

In [ ]:
def run_fastapi():
    uvicorn.run(app, 
                host="0.0.0.0",
                port=8000)

thread = threading.Thread(target=run_fastapi)
thread.start()

INFO:     Started server process [2178]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49875 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:49910 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:49919 - "POST /generate HTTP/1.1" 200 OK


In [9]:
# To stop the server, you can run the following command in your terminal:
#!kill $(lsof -t -i :8000)

In [ ]:
import subprocess
import time
from IPython.display import IFrame, HTML


streamlit_app_code = """
import streamlit as st
import requests

st.title("Demo App")

prompt = st.text_input("Enter your prompt:")
if st.button("Generate Response"):
    if prompt:
        try:
            response = requests.post("http://localhost:8000/generate", json={"prompt": prompt})
            if response.status_code == 200:
                st.success(f"Response: {response.json()['response']}")
            else:
                st.error(f"Error: {response.status_code} - {response.text}")
        except Exception as e:
            st.error(f"Error connecting to FastAPI server: {e}")
    else:
        st.warning("Please enter a prompt before generating a response.")
"""

# Write the app to a temporary file
with open("temp_streamlit_app.py", "w") as f:
    f.write(streamlit_app_code)

# Start Streamlit in the background
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "temp_streamlit_app.py", "--server.port=8501", "--server.headless=true"]
)

# Wait for Streamlit to start
time.sleep(5)

# Embed Streamlit in the notebook
display(IFrame(src="http://127.0.0.1:8501", width="100%", height="600px"))

2026-03-24 10:01:17.091 Port 8501 is already in use


Trace(trace_id=tr-a942d7ec4512af38d1134ab31d025999)

2026-03-24 10:01:29.920 | INFO     | __main__:llm_generate:3 - Generating response for prompt: what is my name
2026-03-24 10:01:33.839 | INFO     | __main__:llm_generate:11 - Generated response: I’m not sure what your name is—could you let me know?
2026-03-24 10:02:03.956 | INFO     | __main__:llm_generate:3 - Generating response for prompt: hello there
2026-03-24 10:02:05.887 | INFO     | __main__:llm_generate:11 - Generated response: Hello! How can I help you today?
